# eICU Sepsis-3 Cohort Selection

This notebook selects patients meeting Sepsis-3 criteria with specific inclusion/exclusion criteria.

**Purpose:** Create a research-ready cohort for Sepsis-3 analysis

**Dataset:** `my-new-project-473015.my_eicu_derived`

---

## Research Design: Cohort Selection Criteria

### Inclusion Criteria
1. ✅ Diagnosis with Sepsis-3 criteria (from `my_eicu_derived.sepsis3`)
2. ✅ First ICU admission with sepsis diagnosis
3. ✅ Age ≥18 years

### Exclusion Criteria
1. ❌ ICU discharge within 24 hours

---

## 📊 Flow Diagram Importance

**Why Flow Diagrams Matter in Research:**

A CONSORT-style flow diagram is essential for:
- **Transparency**: Shows exactly how many patients were excluded at each step
- **Reproducibility**: Others can replicate your cohort selection
- **Quality Control**: Identifies potential data quality issues
- **Publication Requirements**: Most journals require flow diagrams for cohort studies

We will track patient counts at each step to create a complete flow diagram.

---

## 🔄 MIMIC-IV vs eICU: Key Differences

| Aspect | MIMIC-IV | eICU | Notes |
|--------|----------|------|-------|
| **Patient ID** | `subject_id` | `uniquepid` | Unique patient identifier |
| **ICU Stay ID** | `stay_id` | `patientunitstayid` | Unique ICU admission |
| **Hospital ID** | `hadm_id` | `hospitalid` | eICU has multiple hospitals |
| **Age** | `anchor_age` (INT64) | `age` (STRING) | eICU uses "> 89" for elderly |
| **ICU Admission Time** | `intime` (DATETIME) | Offset = 0 | eICU uses offset from admission |
| **ICU Discharge Time** | `outtime` (DATETIME) | `unitdischargeoffset` (minutes) | |
| **LOS Calculation** | `los` column (days) | `unitdischargeoffset / 60 / 24` | Must calculate manually |
| **First Admission** | `ORDER BY intime ASC` | `ORDER BY hospitaladmitoffset ASC` | Smaller (more negative) = earlier |
| **Mortality** | Separate tables | `hospitaldischargestatus`, `unitdischargestatus` | Direct columns |

---

## 🕐 Understanding eICU Time Offsets

In eICU, **ICU admission is the reference point (offset = 0)**:
```
Hospital Admission ──────────────────→ ICU Admission ─────────────→ ICU Discharge
                   ← hospitaladmitoffset →    (0)      → unitdischargeoffset →
                        (NEGATIVE)                           (POSITIVE)
```

**Example:**
| Event | Offset Value | Meaning |
|-------|-------------|---------|
| Hospital admission 3 hours before ICU | -180 | 180 minutes before ICU admission |
| Hospital admission 24 hours before ICU | -1440 | 1440 minutes before ICU admission |
| ICU admission | 0 | Reference point |
| ICU discharge after 48 hours | +2880 | 2880 minutes after ICU admission |

**Key Point:** More negative `hospitaladmitoffset` = Earlier hospital admission

---

## Analysis Steps

**STEP 0:** Setup and configuration  
**STEP 1:** Extract all Sepsis-3 patients  
**STEP 2:** Filter for first ICU admission with sepsis  
**STEP 3:** Apply age filter (≥18 years)  
**STEP 4:** Exclude short ICU stays (<24 hours)  
**STEP 5:** Create final cohort table  
**STEP 6:** Generate flow diagram and cohort statistics

Let's begin!

## STEP 0: Setup


In [71]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

# Initialize BigQuery client
client = bigquery.Client(project='my-new-project-473015')

# Configuration
PROJECT_ID = 'my-new-project-473015'
DATASET_ID = 'my_eicu_derived'

print("="*70)
print("eICU Sepsis-3 Cohort Selection")
print(f"Target dataset: {PROJECT_ID}.{DATASET_ID}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print("\n✅ Setup complete! Ready to select cohort.")
print("\n📊 We will track patient counts at each step for the flow diagram.")

eICU Sepsis-3 Cohort Selection
Target dataset: my-new-project-473015.my_eicu_derived
Timestamp: 2026-01-08 20:21:48

✅ Setup complete! Ready to select cohort.

📊 We will track patient counts at each step for the flow diagram.


## STEP 1: Extract All Sepsis-3 Patients

Extract all patients diagnosed with Sepsis-3 from our previously created table.

**Criteria:** `sepsis3 = TRUE`

**What we extract:**
- Patient identifiers (uniquepid, patientunitstayid)
- Sepsis onset times
- SOFA scores at onset
- Organ dysfunction components

This is our **starting population** for the flow diagram.

### 🔄 Comparison with MIMIC-IV

| MIMIC-IV | eICU |
|----------|------|
| `subject_id` | `uniquepid` |
| `stay_id` | `patientunitstayid` |
| `suspected_infection_time` (DATETIME) | `suspected_infection_time` (minutes offset) |
| `sofa_time` (DATETIME) | `sofa_time` (minutes offset) |

In [72]:
%%bigquery

-- STEP 1: Count all Sepsis-3 patients (starting population)
SELECT
    COUNT(*) AS total_sepsis3_stays,
    COUNT(DISTINCT patientunitstayid) AS unique_icu_stays,
    COUNT(DISTINCT uniquepid) AS unique_patients
FROM `my-new-project-473015.my_eicu_derived.sepsis3`
WHERE sepsis3 = TRUE;

Query is running:   0%|          |

Downloading:   0%|          |

,total_sepsis3_stays,unique_icu_stays,unique_patients
0,43572,43572,36758


In [73]:
%%bigquery step1_result

SELECT
    patientunitstayid,
    uniquepid,
    suspected_infection_time,
    sofa_time,
    sofa_score,
    respiration,
    coagulation,
    liver,
    cardiovascular,
    cns,
    renal,
    sepsis3
FROM `my-new-project-473015.my_eicu_derived.sepsis3`
WHERE sepsis3 = TRUE;

Query is running:   0%|          |

Downloading:   0%|          |

In [74]:
# Count patients
step1_count = len(step1_result)
step1_unique_patients = step1_result['uniquepid'].nunique()
step1_unique_stays = step1_result['patientunitstayid'].nunique()

print("="*70)
print("STEP 1: All Sepsis-3 Patients")
print("="*70)
print(f"Total rows (sepsis events): {step1_count:,}")
print(f"Unique patients: {step1_unique_patients:,}")
print(f"Unique ICU stays: {step1_unique_stays:,}")
print("\n📊 Flow Diagram - Starting Population:")
print(f"   N = {step1_unique_stays:,} ICU stays with Sepsis-3")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step1_result.head())

STEP 1: All Sepsis-3 Patients
Total rows (sepsis events): 43,572
Unique patients: 36,758
Unique ICU stays: 43,572

📊 Flow Diagram - Starting Population:
   N = 43,572 ICU stays with Sepsis-3

📋 Sample data (first 5 rows):
   patientunitstayid   uniquepid  suspected_infection_time  sofa_time  \
0            2487601  022-177778                     -1438          0   
1            2430364   022-65251                     -1434          0   
2             675409  006-243257                     -1434          0   
3            3045435   030-28332                     -1433          0   
4            1691156   017-43412                     -1432          0   

   sofa_score  respiration  coagulation  liver  cardiovascular  cns  renal  \
0           3            2            0      0               1    0      0   
1           2            2            0      0               0    0      0   
2           4            2            0      0               2    0      0   
3           3            0 

## STEP 2: Filter for First ICU Admission with Sepsis

Select only the **first ICU admission** where sepsis was diagnosed for each patient.

**Why this matters:**
- Patients may have multiple ICU admissions
- We want to study the **first sepsis event** to avoid bias from previous treatments
- Uses `ROW_NUMBER()` to rank ICU admissions

### 🔄 Comparison with MIMIC-IV

| Aspect | MIMIC-IV | eICU |
|--------|----------|------|
| **Source Table** | `icustays` | `patient` |
| **Ranking Column** | `intime` (earlier = first) | `hospitaladmitoffset` (more negative = earlier) |
| **Sort Order** | `ORDER BY intime ASC` | `ORDER BY hospitaladmitoffset ASC` |
| **Secondary Sort** | - | `unitvisitnumber ASC` |

**Note on eICU `hospitaladmitoffset`:**
- This is the time from hospital admission to ICU admission (in minutes)
- Values are **typically negative** (hospital admission before ICU admission)
- **More negative** = **earlier** hospital admission
- We sort `ASC` (ascending) so the most negative (earliest) comes first

In [75]:
%%bigquery step2_result

WITH sepsis_patients AS (
    SELECT
        s.patientunitstayid,
        s.uniquepid,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal
    FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
    WHERE s.sepsis3 = TRUE
)
, ranked_admissions AS (
    SELECT
        sp.*,
        p.hospitalid,
        p.wardid,
        p.unittype,
        p.hospitaladmitoffset,
        p.hospitaldischargeoffset,
        p.unitdischargeoffset,
        p.unitvisitnumber,
        -- Rank by hospital admission time
        -- hospitaladmitoffset: more negative = earlier admission
        -- ASC order: most negative (earliest) comes first
        ROW_NUMBER() OVER (
            PARTITION BY sp.uniquepid
            ORDER BY p.hospitaladmitoffset ASC, p.unitvisitnumber ASC
        ) AS admission_rank
    FROM sepsis_patients sp
    INNER JOIN `physionet-data.eicu_crd.patient` p
        ON sp.patientunitstayid = p.patientunitstayid
)
SELECT *
FROM ranked_admissions
WHERE admission_rank = 1;

Query is running:   0%|          |

Downloading:   0%|          |

In [76]:
# Count patients after filtering
step2_count = len(step2_result)
step2_unique_patients = step2_result['uniquepid'].nunique()
step2_unique_stays = step2_result['patientunitstayid'].nunique()

# Calculate exclusions
excluded_step2 = step1_unique_stays - step2_unique_stays

print("="*70)
print("STEP 2: First ICU Admission with Sepsis")
print("="*70)
print(f"Remaining ICU stays: {step2_unique_stays:,}")
print(f"Unique patients: {step2_unique_patients:,}")
print(f"\n❌ Excluded (not first admission): {excluded_step2:,}")
print("\n📊 Flow Diagram Update:")
print(f"   Started with: {step1_unique_stays:,} ICU stays")
print(f"   After first admission filter: {step2_unique_stays:,} ICU stays")
print(f"   Excluded: {excluded_step2:,} ICU stays")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step2_result[['uniquepid', 'patientunitstayid', 'hospitalid', 'hospitaladmitoffset', 'admission_rank', 'sofa_score']].head())

STEP 2: First ICU Admission with Sepsis
Remaining ICU stays: 36,758
Unique patients: 36,758

❌ Excluded (not first admission): 6,814

📊 Flow Diagram Update:
   Started with: 43,572 ICU stays
   After first admission filter: 36,758 ICU stays
   Excluded: 6,814 ICU stays

📋 Sample data (first 5 rows):
   uniquepid  patientunitstayid  hospitalid  hospitaladmitoffset  \
0  002-10009             224606          71                 -179   
1  002-10018             204602          66                -4854   
2  002-10050             221005          71                -2140   
3  002-10052             151900          73                  -23   
4  002-10063             218742          73                   -2   

   admission_rank  sofa_score  
0               1           4  
1               1           3  
2               1           2  
3               1           2  
4               1           2  


## STEP 3: Apply Age Filter (≥18 years)

Exclude patients younger than 18 years old at ICU admission.

**Why this matters:**
- Adult sepsis pathophysiology differs from pediatric
- Treatment protocols are age-specific
- Most clinical trials exclude pediatric patients

### 🔄 Comparison with MIMIC-IV

| Aspect | MIMIC-IV | eICU |
|--------|----------|------|
| **Age Column** | `anchor_age` (INT64) | `age` (STRING) |
| **Age Format** | Numeric (18-91) | String ("0"-"89", "> 89") |
| **Elderly Handling** | 91+ grouped as 91 | "> 89" for patients 90+ |
| **Filter Logic** | `anchor_age >= 18` | `age = '> 89' OR CAST(age AS INT64) >= 18` |

**Note on eICU Age:**
- Age is stored as STRING in eICU
- Patients 90 years and older are coded as "> 89" for privacy
- We convert "> 89" to 90 for numeric analysis
- Must use `SAFE_CAST` to handle non-numeric values

In [77]:
%%bigquery step3_result

WITH first_sepsis_stays AS (
    SELECT
        s.patientunitstayid,
        s.uniquepid,
        ROW_NUMBER() OVER (
            PARTITION BY s.uniquepid
            ORDER BY p.hospitaladmitoffset ASC, p.unitvisitnumber ASC
        ) AS rn
    FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
    INNER JOIN `physionet-data.eicu_crd.patient` p
        ON s.patientunitstayid = p.patientunitstayid
    WHERE s.sepsis3 = TRUE
)
, step2_data AS (
    SELECT
        s.patientunitstayid,
        s.uniquepid,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal,
        p.hospitalid,
        p.wardid,
        p.unittype,
        p.hospitaladmitoffset,
        p.hospitaldischargeoffset,
        p.unitdischargeoffset,
        p.unitvisitnumber,
        p.age AS age_string,
        p.gender,
        p.ethnicity,
        p.hospitaldischargestatus,
        p.unitdischargestatus
    FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
    INNER JOIN `physionet-data.eicu_crd.patient` p
        ON s.patientunitstayid = p.patientunitstayid
    INNER JOIN first_sepsis_stays fs
        ON s.patientunitstayid = fs.patientunitstayid
    WHERE s.sepsis3 = TRUE
        AND fs.rn = 1
)
SELECT
    sd.*,
    -- Convert age string to numeric (handle "> 89" as 90)
    CASE
        WHEN sd.age_string = '> 89' THEN 90
        WHEN SAFE_CAST(sd.age_string AS INT64) IS NOT NULL THEN SAFE_CAST(sd.age_string AS INT64)
        ELSE NULL
    END AS age_numeric
FROM step2_data sd
WHERE
    -- Filter for age >= 18
    (sd.age_string = '> 89' OR SAFE_CAST(sd.age_string AS INT64) >= 18);

Query is running:   0%|          |

Downloading:   0%|          |

In [78]:
# Count patients after age filter
step3_count = len(step3_result)
step3_unique_patients = step3_result['uniquepid'].nunique()
step3_unique_stays = step3_result['patientunitstayid'].nunique()

# Calculate exclusions
excluded_step3 = step2_unique_stays - step3_unique_stays

print("="*70)
print("STEP 3: Age Filter (≥18 years)")
print("="*70)
print(f"Remaining ICU stays: {step3_unique_stays:,}")
print(f"Unique patients: {step3_unique_patients:,}")
print(f"\n❌ Excluded (age <18): {excluded_step3:,}")
print(f"\nAge statistics:")
print(f"   Mean age: {step3_result['age_numeric'].mean():.1f} years")
print(f"   Median age: {step3_result['age_numeric'].median():.1f} years")
print(f"   Min age: {step3_result['age_numeric'].min()} years")
print(f"   Max age: {step3_result['age_numeric'].max()} years")
print("\n📊 Flow Diagram Update:")
print(f"   After first admission filter: {step2_unique_stays:,} ICU stays")
print(f"   After age ≥18 filter: {step3_unique_stays:,} ICU stays")
print(f"   Excluded: {excluded_step3:,} ICU stays")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step3_result[['uniquepid', 'patientunitstayid', 'age_string', 'age_numeric', 'gender', 'sofa_score']].head())


STEP 3: Age Filter (≥18 years)
Remaining ICU stays: 36,725
Unique patients: 36,725

❌ Excluded (age <18): 33

Age statistics:
   Mean age: 65.4 years
   Median age: 67.0 years
   Min age: 18 years
   Max age: 90 years

📊 Flow Diagram Update:
   After first admission filter: 36,758 ICU stays
   After age ≥18 filter: 36,725 ICU stays
   Excluded: 33 ICU stays

📋 Sample data (first 5 rows):
    uniquepid  patientunitstayid age_string  age_numeric gender  sofa_score
0  022-177778            2487601         52           52   Male           3
1   022-65251            2430364         53           53   Male           2
2  006-243257             675409         57           57   Male           4
3   030-28332            3045435         87           87   Male           3
4   017-43412            1691156         51           51   Male           5


### 💡 Note on Pediatric Patients in eICU

**eICU Database Characteristics:**
- eICU includes data from 208 hospitals across the US
- Unlike MIMIC-IV (adult-only), eICU may contain some pediatric patients
- Pediatric exclusions may be non-zero depending on participating hospitals
- Age filtering is important to ensure adult-only cohort


## STEP 4: Exclude Short ICU Stays (<24 hours)

Exclude patients with ICU length of stay <24 hours.

**Rationale:**
- Very short ICU stays may represent:
  - Transfer patients (moved to another facility)
  - Early deaths
  - Misclassification (not truly ICU-level care)
- 24-hour minimum allows observation of clinical trajectory
- Common exclusion criterion in sepsis research

### 🔄 Comparison with MIMIC-IV

| Aspect | MIMIC-IV | eICU |
|--------|----------|------|
| **LOS Source** | `icustays.los` (days) | `patient.unitdischargeoffset` (minutes) |
| **LOS Calculation** | `los * 24` (hours) | `unitdischargeoffset / 60` (hours) |
| **Filter Logic** | `los * 24 >= 24` | `unitdischargeoffset / 60 >= 24` |
| **Precision** | Stored with sub-day precision | Minutes from ICU admission |

**Note on eICU `unitdischargeoffset`:**
- Represents minutes from ICU admission (offset = 0) to discharge
- Always **positive** (discharge is after admission)
- Convert to hours: `unitdischargeoffset / 60`
- Convert to days: `unitdischargeoffset / 60 / 24`

In [79]:
%%bigquery step4_result

WITH first_sepsis_stays AS (
    SELECT
        s.patientunitstayid,
        s.uniquepid,
        ROW_NUMBER() OVER (
            PARTITION BY s.uniquepid
            ORDER BY p.hospitaladmitoffset ASC, p.unitvisitnumber ASC
        ) AS rn
    FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
    INNER JOIN `physionet-data.eicu_crd.patient` p
        ON s.patientunitstayid = p.patientunitstayid
    WHERE s.sepsis3 = TRUE
)
, step3_data AS (
    SELECT
        s.patientunitstayid,
        s.uniquepid,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal,
        p.hospitalid,
        p.wardid,
        p.unittype,
        p.hospitaladmitoffset,
        p.hospitaldischargeoffset,
        p.unitdischargeoffset,
        p.unitvisitnumber,
        p.age AS age_string,
        CASE
            WHEN p.age = '> 89' THEN 90
            WHEN SAFE_CAST(p.age AS INT64) IS NOT NULL THEN SAFE_CAST(p.age AS INT64)
            ELSE NULL
        END AS age_numeric,
        p.gender,
        p.ethnicity,
        p.hospitaldischargestatus,
        p.unitdischargestatus
    FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
    INNER JOIN `physionet-data.eicu_crd.patient` p
        ON s.patientunitstayid = p.patientunitstayid
    INNER JOIN first_sepsis_stays fs
        ON s.patientunitstayid = fs.patientunitstayid
    WHERE s.sepsis3 = TRUE
        AND fs.rn = 1
        AND (p.age = '> 89' OR SAFE_CAST(p.age AS INT64) >= 18)
)
SELECT
    *,
    -- Calculate LOS in hours (unitdischargeoffset is in minutes)
    unitdischargeoffset / 60.0 AS los_hours,
    unitdischargeoffset / 60.0 / 24.0 AS los_days
FROM step3_data
WHERE unitdischargeoffset / 60.0 >= 24;  -- LOS >= 24 hours

Query is running:   0%|          |

Downloading:   0%|          |

In [80]:
# Count final cohort after LOS filter
step4_count = len(step4_result)
step4_unique_patients = step4_result['uniquepid'].nunique()
step4_unique_stays = step4_result['patientunitstayid'].nunique()

# Calculate exclusions
excluded_step4 = step3_unique_stays - step4_unique_stays

print("="*70)
print("STEP 4: Final Results - LOS ≥24 hours")
print("="*70)
print(f"Remaining ICU stays: {step4_unique_stays:,}")
print(f"Unique patients: {step4_unique_patients:,}")
print(f"\n❌ Excluded (LOS <24 hours): {excluded_step4:,}")
print(f"\nLOS Statistics (hours):")
print(f"   Mean: {step4_result['los_hours'].mean():.1f} hours ({step4_result['los_hours'].mean()/24:.1f} days)")
print(f"   Median: {step4_result['los_hours'].median():.1f} hours ({step4_result['los_hours'].median()/24:.1f} days)")
print(f"   Min: {step4_result['los_hours'].min():.1f} hours")
print(f"   Max: {step4_result['los_hours'].max():.1f} hours")
print("\n📊 Flow Diagram Update:")
print(f"   After age ≥18 filter: {step3_unique_stays:,} ICU stays")
print(f"   After LOS ≥24h filter: {step4_unique_stays:,} ICU stays")
print(f"   Excluded: {excluded_step4:,} ICU stays")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step4_result[['uniquepid', 'patientunitstayid', 'age_numeric', 'gender', 'sofa_score', 'los_hours']].head())


STEP 4: Final Results - LOS ≥24 hours
Remaining ICU stays: 31,413
Unique patients: 31,413

❌ Excluded (LOS <24 hours): 5,312

LOS Statistics (hours):
   Mean: 120.5 hours (5.0 days)
   Median: 74.8 hours (3.1 days)
   Min: 24.0 hours
   Max: 2700.6 hours

📊 Flow Diagram Update:
   After age ≥18 filter: 36,725 ICU stays
   After LOS ≥24h filter: 31,413 ICU stays
   Excluded: 5,312 ICU stays

📋 Sample data (first 5 rows):
    uniquepid  patientunitstayid  age_numeric  gender  sofa_score  los_hours
0  022-177778            2487601           52    Male           3  30.950000
1   022-65251            2430364           53    Male           2  97.400000
2   030-28332            3045435           87    Male           3  28.900000
3   017-43412            1691156           51    Male           5  50.516667
4  006-127907             765710           71  Female           3  80.366667


## STEP 5: Create Final Cohort Table

Save the final cohort to a permanent table for future analysis.

**Final Cohort Criteria:**
- ✅ Sepsis-3 diagnosis (SOFA ≥2 + suspected infection)
- ✅ First ICU admission with sepsis
- ✅ Age ≥18 years
- ✅ ICU length of stay ≥24 hours

**Table Name:** `my_eicu_derived.sepsis3_cohort`

### 🔄 Comparison with MIMIC-IV Output Columns

| MIMIC-IV Column | eICU Equivalent | Notes |
|-----------------|-----------------|-------|
| `subject_id` | `uniquepid` | Patient identifier |
| `stay_id` | `patientunitstayid` | ICU stay identifier |
| `hadm_id` | `hospitalid` | Hospital identifier |
| `anchor_age` | `age` | Numeric age |
| `gender` | `gender` | Same |
| `icu_intime` | (offset = 0) | eICU uses offset from admission |
| `icu_outtime` | `unitdischargeoffset` | Minutes from admission |
| `los_days` | `los_days` (calculated) | Same meaning |
| `los_hours` | `los_hours` (calculated) | Same meaning |
| `infection_time_from_icu_admit_hours` | `infection_time_from_icu_admit_hours` | Same meaning |
| (N/A) | `hospital_mortality` | Direct column in eICU |
| (N/A) | `icu_mortality` | Direct column in eICU |

In [81]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_eicu_derived.sepsis3_cohort` AS
WITH first_sepsis_stays AS (
    SELECT
        s.patientunitstayid,
        s.uniquepid,
        ROW_NUMBER() OVER (
            PARTITION BY s.uniquepid
            ORDER BY p.hospitaladmitoffset ASC, p.unitvisitnumber ASC
        ) AS rn
    FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
    INNER JOIN `physionet-data.eicu_crd.patient` p
        ON s.patientunitstayid = p.patientunitstayid
    WHERE s.sepsis3 = TRUE
)
SELECT
    -- Patient identifiers
    s.patientunitstayid,
    s.uniquepid,
    p.hospitalid,

    -- Demographics
    CASE
        WHEN p.age = '> 89' THEN 90
        WHEN SAFE_CAST(p.age AS INT64) IS NOT NULL THEN SAFE_CAST(p.age AS INT64)
        ELSE NULL
    END AS age,
    p.gender,
    p.ethnicity,

    -- ICU stay information
    p.unittype,
    p.unitvisitnumber,
    p.hospitaladmitoffset,
    p.hospitaldischargeoffset,
    p.unitdischargeoffset,
    p.unitdischargeoffset / 60.0 AS los_hours,
    p.unitdischargeoffset / 60.0 / 24.0 AS los_days,

    -- Outcomes
    p.hospitaldischargestatus,
    p.unitdischargestatus,
    CASE WHEN p.hospitaldischargestatus = 'Expired' THEN 1 ELSE 0 END AS hospital_mortality,
    CASE WHEN p.unitdischargestatus = 'Expired' THEN 1 ELSE 0 END AS icu_mortality,

    -- Sepsis onset information
    s.suspected_infection_time,
    s.sofa_time,
    s.suspected_infection_time / 60.0 AS infection_time_from_icu_admit_hours,

    -- SOFA scores
    s.sofa_score AS sofa_at_sepsis_onset,
    s.respiration AS sofa_respiration,
    s.coagulation AS sofa_coagulation,
    s.liver AS sofa_liver,
    s.cardiovascular AS sofa_cardiovascular,
    s.cns AS sofa_cns,
    s.renal AS sofa_renal,

    -- Inclusion/exclusion tracking
    'Included' AS cohort_status,
    CURRENT_TIMESTAMP() AS cohort_creation_time

FROM `my-new-project-473015.my_eicu_derived.sepsis3` s
INNER JOIN `physionet-data.eicu_crd.patient` p
    ON s.patientunitstayid = p.patientunitstayid
INNER JOIN first_sepsis_stays fs
    ON s.patientunitstayid = fs.patientunitstayid
WHERE s.sepsis3 = TRUE
    AND fs.rn = 1
    AND (p.age = '> 89' OR SAFE_CAST(p.age AS INT64) >= 18)
    AND p.unitdischargeoffset / 60.0 >= 24;

Query is running:   0%|          |

""


In [82]:
print("✅ Final cohort table created: my_eicu_derived.sepsis3_cohort")


✅ Final cohort table created: my_eicu_derived.sepsis3_cohort


## STEP 6: Verify Final Cohort and Generate Statistics


In [83]:
%%bigquery cohort_stats

-- Verify final cohort
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT patientunitstayid) AS unique_icu_stays,
    COUNT(DISTINCT uniquepid) AS unique_patients,
    ROUND(AVG(age), 1) AS mean_age,
    ROUND(AVG(los_hours), 1) AS mean_los_hours,
    ROUND(AVG(los_days), 1) AS mean_los_days,
    ROUND(AVG(sofa_at_sepsis_onset), 2) AS mean_sofa,
    SUM(hospital_mortality) AS hospital_deaths,
    ROUND(100.0 * SUM(hospital_mortality) / COUNT(*), 1) AS hospital_mortality_pct,
    SUM(icu_mortality) AS icu_deaths,
    ROUND(100.0 * SUM(icu_mortality) / COUNT(*), 1) AS icu_mortality_pct
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort`;

Query is running:   0%|          |

Downloading:   0%|          |

In [84]:
print("="*70)
print("FINAL COHORT STATISTICS")
print("="*70)
print(cohort_stats.to_string(index=False))
print("="*70)

FINAL COHORT STATISTICS
 total_rows  unique_icu_stays  unique_patients  mean_age  mean_los_hours  mean_los_days  mean_sofa  hospital_deaths  hospital_mortality_pct  icu_deaths  icu_mortality_pct
      31410             31410            31410      65.2           120.4            5.0       3.57             4371                    13.9        2956                9.4


In [85]:
%%bigquery gender_dist

-- Gender distribution
SELECT
    gender,
    COUNT(*) AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS percentage
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort`
GROUP BY gender
ORDER BY count DESC;

Query is running:   0%|          |

Downloading:   0%|          |

In [86]:
%%bigquery unittype_dist

-- Unit type distribution (eICU-specific)
SELECT
    unittype,
    COUNT(*) AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS percentage
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort`
GROUP BY unittype
ORDER BY count DESC;

Query is running:   0%|          |

Downloading:   0%|          |

In [87]:
%%bigquery sofa_dist

-- SOFA score distribution
SELECT
    sofa_at_sepsis_onset,
    COUNT(*) AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS percentage
FROM `my-new-project-473015.my_eicu_derived.sepsis3_cohort`
GROUP BY sofa_at_sepsis_onset
ORDER BY sofa_at_sepsis_onset;

Query is running:   0%|          |

Downloading:   0%|          |

## Flow Diagram Summary


In [88]:
print("="*80)
print("                    SEPSIS-3 COHORT SELECTION FLOW DIAGRAM")
print("="*80)
print()
print("┌─────────────────────────────────────────────────────────────────────────┐")
print("│                                                                         │")
print(f"│  eICU Database: All Sepsis-3 Patients                                  │")
print(f"│  N = {step1_unique_stays:,} ICU stays                                          │")
print("│  (Sepsis-3 criteria: SOFA ≥2 + suspected infection)                    │")
print("│                                                                         │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print(f"│  STEP 2: Filter for FIRST ICU admission with sepsis                    │")
print(f"│  Remaining: {step2_unique_stays:,} ICU stays                                   │")
print(f"│  Excluded: {step1_unique_stays - step2_unique_stays:,} (not first admission)                              │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print(f"│  STEP 3: Apply age filter (≥18 years)                                  │")
print(f"│  Remaining: {step3_unique_stays:,} ICU stays                                   │")
print(f"│  Excluded: {step2_unique_stays - step3_unique_stays:,} (age <18 years)                                     │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print(f"│  STEP 4: Exclude short ICU stays (<24 hours)                           │")
print(f"│  Remaining: {step4_unique_stays:,} ICU stays                                   │")
print(f"│  Excluded: {step3_unique_stays - step4_unique_stays:,} (LOS <24 hours)                              │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("╔═════════════════════════════════════════════════════════════════════════╗")
print("║                                                                         ║")
print("║                    FINAL SEPSIS-3 COHORT                                ║")
print(f"║                    N = {step4_unique_stays:,} patients                              ║")
print("║                                                                         ║")
print("╚═════════════════════════════════════════════════════════════════════════╝")
print()
print("="*80)

                    SEPSIS-3 COHORT SELECTION FLOW DIAGRAM

┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  eICU Database: All Sepsis-3 Patients                                  │
│  N = 43,572 ICU stays                                          │
│  (Sepsis-3 criteria: SOFA ≥2 + suspected infection)                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
                                  │
                                  │
                                  ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 2: Filter for FIRST ICU admission with sepsis                    │
│  Remaining: 36,758 ICU stays                                   │
│  Excluded: 6,814 (not first admission)                              │
└────────────────────

## Exclusion Summary Table

In [89]:
import pandas as pd

# Create exclusion summary
exclusion_data = {
    'Step': [
        'Starting population',
        'After first admission filter',
        'After age ≥18 filter',
        'After LOS ≥24h filter',
        'FINAL COHORT'
    ],
    'Remaining (N)': [
        f"{step1_unique_stays:,}",
        f"{step2_unique_stays:,}",
        f"{step3_unique_stays:,}",
        f"{step4_unique_stays:,}",
        f"{step4_unique_stays:,}"
    ],
    'Excluded (N)': [
        '-',
        f"{step1_unique_stays - step2_unique_stays:,}",
        f"{step2_unique_stays - step3_unique_stays:,}",
        f"{step3_unique_stays - step4_unique_stays:,}",
        '-'
    ],
    'Exclusion Reason': [
        'All Sepsis-3 patients',
        'Not first ICU admission',
        'Age <18 years',
        'ICU LOS <24 hours',
        'Met all criteria'
    ]
}

exclusion_df = pd.DataFrame(exclusion_data)

print("="*80)
print("EXCLUSION SUMMARY TABLE")
print("="*80)
print(exclusion_df.to_string(index=False))
print("="*80)

EXCLUSION SUMMARY TABLE
                        Step Remaining (N) Excluded (N)        Exclusion Reason
         Starting population        43,572            -   All Sepsis-3 patients
After first admission filter        36,758        6,814 Not first ICU admission
        After age ≥18 filter        36,725           33           Age <18 years
       After LOS ≥24h filter        31,413        5,312       ICU LOS <24 hours
                FINAL COHORT        31,413            -        Met all criteria


## 📋 Summary: MIMIC-IV vs eICU Implementation

| Step | MIMIC-IV Implementation | eICU Implementation |
|------|------------------------|---------------------|
| **STEP 1** | `SELECT FROM sepsis3 WHERE sepsis3 = TRUE` | Same |
| **STEP 2** | `ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime ASC)` | `ROW_NUMBER() OVER (PARTITION BY uniquepid ORDER BY hospitaladmitoffset ASC, unitvisitnumber ASC)` |
| **STEP 3** | `WHERE anchor_age >= 18` | `WHERE age = '> 89' OR CAST(age AS INT64) >= 18` |
| **STEP 4** | `WHERE los * 24 >= 24` | `WHERE unitdischargeoffset / 60 >= 24` |
| **STEP 5** | Create `sepsis3_cohort` table | Same |
| **Output** | `subject_id`, `stay_id`, `hadm_id` | `uniquepid`, `patientunitstayid`, `hospitalid` |